# Ejercicio 5. ¿Puede un ESP32 usar un LLM? ¿Qué restricciones tendría?

**Enunciado:** Suponer una aplicación con un microcontrolador como ESP32: ¿podría usar un LLM y qué restricciones tendría?

Fuente: Martínez P. (2026), sección 7.11.

## Cómo utilizar este notebook

Abre el archivo en Jupyter o VS Code con un kernel Python del mismo entorno del proyecto. Colócalo en la raíz o en `notebooks/`. Las celdas de texto contienen la respuesta al ejercicio; las de código generan evidencia al ejecutarlas en tu computador.

No se han ejecutado las consultas contra tu Ollama ni tu base de datos. Los resultados esperados no se presentan como resultados obtenidos. Si hace falta instalar Jupyter, utiliza en tu entorno `python -m pip install notebook ipykernel`; las dependencias de la aplicación se instalan desde `requirements.txt`.

Las rutas y funciones corresponden al código desarrollado en la conversación. Los manuales concretos, los modelos instalados y el contenido real de tus skills deben verificarse localmente.

## Respuesta

**Sí puede utilizar un LLM alojado en otro equipo**, enviando datos o consultas por red y recibiendo respuestas. Esto debe distinguirse de ejecutar localmente un modelo convencional de miles de millones de parámetros.

Para este proyecto se propone que el ESP32 lea sensores y envíe datos por HTTP o MQTT a un servicio en el computador. Ese servicio validaría los datos y consultaría el sistema Python con Ollama; el ESP32 recibiría una respuesta para mostrar un aviso. Esta integración es una propuesta conceptual: no se ha desarrollado firmware ni un servidor HTTP/MQTT en el código entregado.

| Restricción | Implicación |
|---|---|
| RAM y almacenamiento | Los pesos, el contexto y la memoria de ejecución de un LLM convencional exceden normalmente los recursos de un microcontrolador |
| Capacidad de cómputo | La generación local de texto con modelos grandes no resulta práctica en un ESP32 convencional |
| Conectividad | La consulta externa depende de que el servidor y la red estén disponibles |
| Latencia | La respuesta del LLM no ofrece tiempos deterministas para control en tiempo real |
| Energía | La comunicación frecuente y el procesamiento compiten con el presupuesto energético del dispositivo |
| Fiabilidad | Las respuestas pueden ser incorrectas; deben validarse antes de utilizarlas |

Como referencia concreta, Espressif describe el ESP32-S3 con unos cientos de kilobytes de RAM interna y soporte de PSRAM externa, sujeto a restricciones (Espressif Systems, s. f.). Esa capacidad no debe generalizarse a todas las placas ESP32 ni confundirse con memoria externa instalada de fábrica.

Como estimación propia, un modelo de 1.000 millones de parámetros a 4 bits requiere aproximadamente 500 MB solo para sus pesos: `1.000.000.000 × 4 / 8` bytes. A esto se suman otros consumos de memoria. Por tanto, para un modelo de ese tamaño es más razonable ejecutar la inferencia en el computador y utilizar el ESP32 como dispositivo de adquisición y comunicación. Esto no excluye modelos mucho más pequeños y específicos de IA local.

Las protecciones, alarmas críticas y control de actuadores deben conservar una lógica determinista independiente de la respuesta del LLM. En esta propuesta el modelo apoya la explicación y consulta documental.

## Ejemplo de aplicación propuesto

Un sensor mide temperatura; el ESP32 envía una lectura con su unidad y código de equipo a un servicio en el computador. El servicio valida formato y valores, consulta el sistema Python y devuelve un texto para una pantalla. El firmware y ese servicio intermedio siguen pendientes.

`config.py` utiliza `localhost` porque Python y Ollama se ejecutan en el mismo computador. En el ESP32, `localhost` sería el propio microcontrolador: no permite acceder al computador. Haría falta configurar la dirección del servicio en la red, autenticación y manejo de desconexiones. No se propone exponer Ollama directamente a Internet.

## Estimación reproducible de memoria

La siguiente celda usa solamente Python y no requiere el proyecto ni hardware. Estima pesos sin contar escalas de cuantización, caché de atención, activaciones o buffers; no mide el consumo real de un modelo.

In [ ]:
def estimate_weight_memory(parameters, bits_per_parameter):
    if parameters <= 0 or bits_per_parameter <= 0:
        raise ValueError("Los valores deben ser positivos.")
    size_bytes = parameters * bits_per_parameter / 8
    return {"MB_decimales": size_bytes / 1_000_000,
            "MiB": size_bytes / (1024 ** 2)}

for parameters in [100_000_000, 1_000_000_000, 3_000_000_000]:
    memory = estimate_weight_memory(parameters, 4)
    print(f"{parameters:,} parámetros a 4 bits: "
          f"{memory['MB_decimales']:.1f} MB / {memory['MiB']:.1f} MiB")

## Datos ilustrativos y respuesta esperada

Los siguientes datos son ficticios, no lecturas de sensores ni umbrales de alarma. No permiten diagnosticar una falla sin límites del fabricante y contexto operativo.

In [ ]:
import json

telemetry_example = {
    "equipo": "DEMO-M-101",
    "temperatura": {"valor": 72.0, "unidad": "°C"},
    "origen": "ejemplo ficticio, no hardware",
}
print(json.dumps(telemetry_example, ensure_ascii=False, indent=2))

## Conclusión del ejercicio

El ESP32 puede consumir un servicio que utilice un LLM externo. Ejecutar un LLM convencional grande en el propio microcontrolador no resulta práctico por sus recursos. Las tareas pequeñas de IA local son un caso distinto. El control crítico debe mantener decisiones deterministas independientes de las respuestas generadas; esta propuesta limita el LLM al apoyo documental y explicativo.

## Referencias

AI Hero. (s. f.). *Become a real AI Hero*. https://www.aihero.dev/

Chroma. (s. f.-a). *Configure collections*. https://docs.trychroma.com/docs/collections/configure

Chroma. (s. f.-b). *Query and get*. https://docs.trychroma.com/docs/querying-collections/query-and-get

Espressif Systems. (s. f.). *Support for external RAM: ESP32-S3*. ESP-IDF Programming Guide. https://docs.espressif.com/projects/esp-idf/en/stable/esp32s3/api-guides/external-ram.html

Martínez P., J. J. (2026, septiembre). *7. Inteligencia artificial generativa* [Material de curso proporcionado en el archivo `7.md`].

Ollama. (s. f.-a). *Generate embeddings*. https://docs.ollama.com/api/embed

Ollama. (s. f.-b). *Tool calling*. https://docs.ollama.com/capabilities/tool-calling

pypdf. (s. f.). *Extract text from a PDF*. https://pypdf.readthedocs.io/en/stable/user/extract-text.html